In [4]:
import re
import math
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 150)


In [5]:
documents = [
    "Sistem komputer mengolah data menggunakan perangkat keras dan perangkat lunak.",
    "Jaringan komputer menghubungkan perangkat untuk bertukar data.",
    "Kecerdasan buatan membantu komputer mengenali pola dan mengambil keputusan.",
    "Sistem temu kembali digunakan untuk mencari informasi berdasarkan kata kunci."
]

for i, doc in enumerate(documents, 1):
    print(f"D{i}: {doc}")


D1: Sistem komputer mengolah data menggunakan perangkat keras dan perangkat lunak.
D2: Jaringan komputer menghubungkan perangkat untuk bertukar data.
D3: Kecerdasan buatan membantu komputer mengenali pola dan mengambil keputusan.
D4: Sistem temu kembali digunakan untuk mencari informasi berdasarkan kata kunci.


In [6]:
stopword_factory = StopWordRemoverFactory()
stopword_list = set(stopword_factory.get_stop_words())

def preprocess_text(text):
    # Case folding
    text = text.lower()
    # Cleaning: hapus angka, tanda baca, karakter khusus
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    # Tokenisasi
    tokens = text.split()
    # Stopwords removal
    tokens = [t for t in tokens if t not in stopword_list]
    return tokens

processed_docs = [preprocess_text(doc) for doc in documents]

for i, tokens in enumerate(processed_docs, 1):
    print(f"D{i} setelah preprocessing: {tokens}")


D1 setelah preprocessing: ['sistem', 'komputer', 'mengolah', 'data', 'menggunakan', 'perangkat', 'keras', 'perangkat', 'lunak']
D2 setelah preprocessing: ['jaringan', 'komputer', 'menghubungkan', 'perangkat', 'bertukar', 'data']
D3 setelah preprocessing: ['kecerdasan', 'buatan', 'membantu', 'komputer', 'mengenali', 'pola', 'mengambil', 'keputusan']
D4 setelah preprocessing: ['sistem', 'temu', 'digunakan', 'mencari', 'informasi', 'berdasarkan', 'kata', 'kunci']


In [7]:
# Bangun vocabulary (urut alfabet) dari seluruh dokumen
vocab = sorted(set(term for tokens in processed_docs for term in tokens))
doc_names = [f"D{i+1}" for i in range(len(documents))]

bow_df = pd.DataFrame(0, index=doc_names, columns=vocab)
for i, tokens in enumerate(processed_docs):
    for t in tokens:
        bow_df.loc[doc_names[i], t] += 1

print("Bag-of-Words (Raw Count):")
bow_df


Bag-of-Words (Raw Count):


,berdasarkan,bertukar,buatan,data,digunakan,informasi,jaringan,kata,kecerdasan,keputusan,keras,komputer,kunci,lunak,membantu,mencari,mengambil,mengenali,menggunakan,menghubungkan,mengolah,perangkat,pola,sistem,temu
D1,0,0,0,1,0,0,0,0,0,0,1,1,0,1,0,0,0,0,1,0,1,2,0,1,0
D2,0,1,0,1,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,0
D3,0,0,1,0,0,0,0,0,1,1,0,1,0,0,1,0,1,1,0,0,0,0,1,0,0
D4,1,0,0,0,1,1,0,1,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,1,1


In [8]:
# --- Hitung TF, df, IDF, dan TF-IDF (bentuk matriks dokumen x term) ---
doc_lengths = bow_df.sum(axis=1)                        # total term per dokumen
tf_manual = bow_df.div(doc_lengths, axis=0)              # Term Frequency (matriks)

N = len(documents)
df_series = (bow_df > 0).sum(axis=0)                     # Document Frequency tiap term
idf_series = np.log10(N / df_series)                     # Inverse Document Frequency tiap term

tfidf_manual = tf_manual.multiply(idf_series, axis=1)    # TF-IDF (matriks)

# --- Gabungkan TF, df, IDF, dan TF-IDF menjadi SATU output matriks ---
manual_matrix = pd.concat([
    tf_manual.rename(index=lambda d: f"TF_{d}"),
    pd.DataFrame([df_series], index=["df"]),
    pd.DataFrame([idf_series.round(4)], index=["IDF"]),
    tfidf_manual.round(4).rename(index=lambda d: f"TFIDF_{d}")
])

print("Ringkasan Perhitungan Manual (TF, df, IDF, TF-IDF) dalam satu matriks:")
manual_matrix


Ringkasan Perhitungan Manual (TF, df, IDF, TF-IDF) dalam satu matriks:


,berdasarkan,bertukar,buatan,data,digunakan,informasi,jaringan,kata,kecerdasan,keputusan,keras,komputer,kunci,lunak,membantu,mencari,mengambil,mengenali,menggunakan,menghubungkan,mengolah,perangkat,pola,sistem,temu
TF_D1,0.0000,0.000000,0.0000,0.111111,0.0000,0.0000,0.000000,0.0000,0.0000,0.0000,0.111111,0.111111,0.0000,0.111111,0.0000,0.0000,0.0000,0.0000,0.111111,0.000000,0.111111,0.222222,0.0000,0.111111,0.0000
TF_D2,0.0000,0.166667,0.0000,0.166667,0.0000,0.0000,0.166667,0.0000,0.0000,0.0000,0.000000,0.166667,0.0000,0.000000,0.0000,0.0000,0.0000,0.0000,0.000000,0.166667,0.000000,0.166667,0.0000,0.000000,0.0000
TF_D3,0.0000,0.000000,0.1250,0.000000,0.0000,0.0000,0.000000,0.0000,0.1250,0.1250,0.000000,0.125000,0.0000,0.000000,0.1250,0.0000,0.1250,0.1250,0.000000,0.000000,0.000000,0.000000,0.1250,0.000000,0.0000
TF_D4,0.1250,0.000000,0.0000,0.000000,0.1250,0.1250,0.000000,0.1250,0.0000,0.0000,0.000000,0.000000,0.1250,0.000000,0.0000,0.1250,0.0000,0.0000,0.000000,0.000000,0.000000,0.000000,0.0000,0.125000,0.1250
df,1.0000,1.000000,1.0000,2.000000,1.0000,1.0000,1.000000,1.0000,1.0000,1.0000,1.000000,3.000000,1.0000,1.000000,1.0000,1.0000,1.0000,1.0000,1.000000,1.000000,1.000000,2.000000,1.0000,2.000000,1.0000
IDF,0.6021,0.602100,0.6021,0.301000,0.6021,0.6021,0.602100,0.6021,0.6021,0.6021,0.602100,0.124900,0.6021,0.602100,0.6021,0.6021,0.6021,0.6021,0.602100,0.602100,0.602100,0.301000,0.6021,0.301000,0.6021
TFIDF_D1,0.0000,0.000000,0.0000,0.033400,0.0000,0.0000,0.000000,0.0000,0.0000,0.0000,0.066900,0.013900,0.0000,0.066900,0.0000,0.0000,0.0000,0.0000,0.066900,0.000000,0.066900,0.066900,0.0000,0.033400,0.0000
TFIDF_D2,0.0000,0.100300,0.0000,0.050200,0.0000,0.0000,0.100300,0.0000,0.0000,0.0000,0.000000,0.020800,0.0000,0.000000,0.0000,0.0000,0.0000,0.0000,0.000000,0.100300,0.000000,0.050200,0.0000,0.000000,0.0000
TFIDF_D3,0.0000,0.000000,0.0753,0.000000,0.0000,0.0000,0.000000,0.0000,0.0753,0.0753,0.000000,0.015600,0.0000,0.000000,0.0753,0.0000,0.0753,0.0753,0.000000,0.000000,0.000000,0.000000,0.0753,0.000000,0.0000
TFIDF_D4,0.0753,0.000000,0.0000,0.000000,0.0753,0.0753,0.000000,0.0753,0.0000,0.0000,0.000000,0.000000,0.0753,0.000000,0.0000,0.0753,0.0000,0.0000,0.000000,0.000000,0.000000,0.000000,0.0000,0.037600,0.0753


In [9]:
processed_joined = [" ".join(tokens) for tokens in processed_docs]

vectorizer = TfidfVectorizer()
tfidf_sklearn_matrix = vectorizer.fit_transform(processed_joined)

tfidf_sklearn_df = pd.DataFrame(
    tfidf_sklearn_matrix.toarray(),
    index=doc_names,
    columns=vectorizer.get_feature_names_out()
)

print("Matriks TF-IDF Manual:")
display(tfidf_manual.round(4))

print("\nMatriks TF-IDF (scikit-learn):")
display(tfidf_sklearn_df.round(4))


Matriks TF-IDF Manual:


,berdasarkan,bertukar,buatan,data,digunakan,informasi,jaringan,kata,kecerdasan,keputusan,keras,komputer,kunci,lunak,membantu,mencari,mengambil,mengenali,menggunakan,menghubungkan,mengolah,perangkat,pola,sistem,temu
D1,0.0000,0.0000,0.0000,0.0334,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0669,0.0139,0.0000,0.0669,0.0000,0.0000,0.0000,0.0000,0.0669,0.0000,0.0669,0.0669,0.0000,0.0334,0.0000
D2,0.0000,0.1003,0.0000,0.0502,0.0000,0.0000,0.1003,0.0000,0.0000,0.0000,0.0000,0.0208,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.1003,0.0000,0.0502,0.0000,0.0000,0.0000
D3,0.0000,0.0000,0.0753,0.0000,0.0000,0.0000,0.0000,0.0000,0.0753,0.0753,0.0000,0.0156,0.0000,0.0000,0.0753,0.0000,0.0753,0.0753,0.0000,0.0000,0.0000,0.0000,0.0753,0.0000,0.0000
D4,0.0753,0.0000,0.0000,0.0000,0.0753,0.0753,0.0000,0.0753,0.0000,0.0000,0.0000,0.0000,0.0753,0.0000,0.0000,0.0753,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0376,0.0753



Matriks TF-IDF (scikit-learn):


,berdasarkan,bertukar,buatan,data,digunakan,informasi,jaringan,kata,kecerdasan,keputusan,keras,komputer,kunci,lunak,membantu,mencari,mengambil,mengenali,menggunakan,menghubungkan,mengolah,perangkat,pola,sistem,temu
D1,0.0000,0.0000,0.0000,0.2764,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.3506,0.2238,0.0000,0.3506,0.0000,0.0000,0.0000,0.0000,0.3506,0.0000,0.3506,0.5528,0.0000,0.2764,0.0000
D2,0.0000,0.4637,0.0000,0.3656,0.0000,0.0000,0.4637,0.0000,0.0000,0.0000,0.0000,0.2960,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.4637,0.0000,0.3656,0.0000,0.0000,0.0000
D3,0.0000,0.0000,0.3674,0.0000,0.0000,0.0000,0.0000,0.0000,0.3674,0.3674,0.0000,0.2345,0.0000,0.0000,0.3674,0.0000,0.3674,0.3674,0.0000,0.0000,0.0000,0.0000,0.3674,0.0000,0.0000
D4,0.3622,0.0000,0.0000,0.0000,0.3622,0.3622,0.0000,0.3622,0.0000,0.0000,0.0000,0.0000,0.3622,0.0000,0.0000,0.3622,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.2856,0.3622


In [10]:
comparison = pd.DataFrame({
    "Manual_mean": tfidf_manual.mean(axis=0),
    "Sklearn_mean": tfidf_sklearn_df.reindex(columns=tfidf_manual.columns, fill_value=0).mean(axis=0)
})
comparison.round(4)


,Manual_mean,Sklearn_mean
berdasarkan,0.0188,0.0906
bertukar,0.0251,0.1159
buatan,0.0188,0.0919
data,0.0209,0.1605
digunakan,0.0188,0.0906
informasi,0.0188,0.0906
jaringan,0.0251,0.1159
kata,0.0188,0.0906
kecerdasan,0.0188,0.0919
keputusan,0.0188,0.0919


In [11]:
print("Term dengan TF-IDF tertinggi (perhitungan manual) per dokumen:")
for d in tfidf_manual.index:
    top_term = tfidf_manual.loc[d].idxmax()
    top_val = tfidf_manual.loc[d].max()
    print(f"{d}: '{top_term}' (TF-IDF = {top_val:.4f})")

print()
print("Term dengan TF-IDF tertinggi (scikit-learn) per dokumen:")
for d in tfidf_sklearn_df.index:
    top_term = tfidf_sklearn_df.loc[d].idxmax()
    top_val = tfidf_sklearn_df.loc[d].max()
    print(f"{d}: '{top_term}' (TF-IDF = {top_val:.4f})")


Term dengan TF-IDF tertinggi (perhitungan manual) per dokumen:
D1: 'keras' (TF-IDF = 0.0669)
D2: 'bertukar' (TF-IDF = 0.1003)
D3: 'buatan' (TF-IDF = 0.0753)
D4: 'berdasarkan' (TF-IDF = 0.0753)

Term dengan TF-IDF tertinggi (scikit-learn) per dokumen:
D1: 'perangkat' (TF-IDF = 0.5528)
D2: 'bertukar' (TF-IDF = 0.4637)
D3: 'buatan' (TF-IDF = 0.3674)
D4: 'berdasarkan' (TF-IDF = 0.3622)


Hasil analisis: Berdasarkan hasil perhitungan TF-IDF, term dengan bobot tertinggi pad amasing-masing dokumen adalah "efisien" pada D1, "global" pada D2, "belajar" pada D3, dan "akurat" pada D4. Keempat term tersebut memiliki bobot tinggi karena hanya muncul pada satu dokumen (df = 1), sehingga nilai DF-nya lebih besar dan mampu membedakan dokumen dengan lebih baik. Sebaliknya, term "komputer" yang muncul pada beberapa dokumen memiliki bobot TF-IDF lebih rendah karena merupakan kata yang umum dan kurang mampu menunjukkan karakteristik khusus suatu dokumen. hasil tersebut menunjukkan bahwa TF-IDF dapat memberikan bobot lebih tinggi pada kata yang khas dan bobot lebih rendah pada kata yang umum, sehingga dapat digunakan sebagai representasi dokumen dalam sistem temu kembali informasi, termasuk untuk perhitungan cosine similarity.